In [ ]:
# Importar as bibliotecas

import datetime
import json
import os
import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
import sys
import matplotlib.pyplot as plt

In [ ]:
# Resolver os 'imports' do projeto 

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from utils.classes.pandas_dataframe import PandasDataframe as pd_dataframe

In [ ]:
# Trazer as opções de configuração do JSON

json_file = os.path.abspath('../../options.json')

with open(json_file, 'r', encoding='utf-8') as j_file:
    json_data = json.load(j_file)

In [ ]:
# Declaração dos caminhos

dir_archive = os.path.abspath('../../data')

dir_cvm = os.path.join(dir_archive, 'cvm')
dir_cdi = os.path.join(dir_archive, 'cdi')

dir_cvm_processed = os.path.join(dir_cvm, json_data['DIR']['CVM']['DATA_NAME'])

In [ ]:
# Manejo da data dos investimentos

START_DATE = json_data['CONFIG']['START_DATE']
END_DATE = json_data.get("CONFIG", False).get("END_DATE", False) or str(datetime.date.today())

In [ ]:
# Gráfico de valorização

cvm_list = json_data['CHARTS']['INVESTMENT_LIST']
metric_list = json_data['CHARTS']['METRICS_LIST']

CSV_EXTENSION = '.csv'

def transforming_archives_in_df():
    cvm_archives = []
    for cvm_cnpj in cvm_list:
        cvm_archive_name = json_data['INVESTMENTS'][cvm_cnpj]['INVESTMENT_FUND']
        cvm_archive_name = cvm_archive_name.replace(' ', '_')

        cvm_archive_path = os.path.join(dir_cvm, json_data['DIR']['CVM']['DATA_NAME'], cvm_archive_name + CSV_EXTENSION)
        cvm_archive = pd_dataframe(cvm_archive_path, None)
        cvm_archive.csv_to_df() 
    
        cvm_archives.append(cvm_archive.df)

    metrics_archives = []
    for metric_key_name in metric_list:
        metric_archive_path = os.path.join(dir_archive, metric_key_name.lower(), metric_key_name.lower() + '_valuation' + CSV_EXTENSION)
        metric_archive = pd_dataframe(metric_archive_path, None)
        metric_archive.csv_to_df()

        metrics_archives.append(metric_archive.df)

    return [ cvm_archives, metrics_archives ]

def flateness_list(data_lists):
    flat_list = []
    for data_list in data_lists:
        for data in data_list:
            flat_list.append(data)
    return flat_list

def compile_in_dataframe(cvm_archives, metric_archives):
    [name_data, valuation_data, date_data] = [[], [], []]

    for cvm_archive in cvm_archives:
        cvm_archive = pd_dataframe(None, cvm_archive)

        cvm_name = cvm_archive.get_column_in_list('NOME_FUNDOS')
        name_data.append(cvm_name)

        cvm_valuation = cvm_archive.get_column_in_list('VALUATION_PERCENT')
        valuation_data.append(cvm_valuation)

        cvm_date = cvm_archive.get_column_in_list('DT_COMPTC')
        date_data.append(cvm_date)

    for metric_archive in metric_archives:
        metric_archive = pd_dataframe(None, metric_archive)

        metric_name = metric_archive.get_column_in_list('METRIC_NAME')
        name_data.append(metric_name)

        metric_valuation = metric_archive.get_column_in_list('VALUATION_PERCENT')
        valuation_data.append(metric_valuation)

        metric_date = metric_archive.get_column_in_list('DATE')
        date_data.append(metric_date)

    name_data = flateness_list(name_data)
    valuation_data = flateness_list(valuation_data)
    date_data = flateness_list(date_data)

    object_valuation_data = {
        'NAME': name_data,
        'VALUATION_PERCENT': valuation_data,
        'DATE': date_data,
    }

    valuation_df = pd_dataframe(None, None, dict=object_valuation_data)
    valuation_df.dict_to_df()

    return valuation_df

def order_by_date(valuation_df):
    valuation_df.df['DATE'] = pd.to_datetime(valuation_df.df['DATE'])
    valuation_df.sort_elements_list(['DATE'])
    return valuation_df

def processing_graph_valuation_data():
    [cvm_archives, metrics_archives] = transforming_archives_in_df()
    valuation_df = compile_in_dataframe(cvm_archives, metrics_archives)
    valuation_df = order_by_date(valuation_df)
    return valuation_df


In [ ]:
data_valuation = processing_graph_valuation_data()

sns.set_style("whitegrid")
plt.figure(figsize=(20, 6))

plot = sns.lineplot(
    data=data_valuation.df, 
    x='DATE', 
    y='VALUATION_PERCENT', 
    hue='NAME', 
    linewidth=2.5
)

plt.title("Evolução do percentual de valorização", fontsize=16, fontweight='bold', pad=20)
plt.text(
    1.225, 0.0,
    "Atualizado em " + str(datetime.datetime.today().date()), 
    transform=plt.gca().transAxes, 
    fontsize=10, 
    va='bottom', ha='left'
)

plt.xlabel("Período", fontsize=14)
plt.ylabel("Percentual de valorização (%)", fontsize=14)
plt.xticks(
    rotation=45, 
    fontsize=11,
    ha='right',
    rotation_mode= 'anchor'
)
plt.legend(
    title='Fundo de Investimento', 
    bbox_to_anchor=(1.05, 1), 
    fontsize=11
)

plt.gca().tick_params(axis='x', pad=5)
plt.show()